# NOTEBOOK 05 — NLP PREPROCESSING

## Project
Job Market Analyzer for Career Readiness in South Africa

## Purpose
Preprocess job description text and skills for NLP analysis, TF-IDF vectorization, and similarity calculations.

## Position in Pipeline

Previous: Notebook 04 — EDA

Current: Notebook 05

Next: Notebook 06 — Feature Engineering

## Main Objective
Transform raw text into clean, normalized tokens suitable for machine learning and similarity analysis.

## Input
- `data/interim/sa_it_jobs_clean.csv` — cleaned dataset

## Output
- Processed text columns
- TF-IDF top terms visualization
- Text length distributions

---

# NOTEBOOK ROADMAP

STEP 1 — Define NLP Objective
STEP 2 — Load Job Text
STEP 3 — Inspect Raw Text
STEP 4 — Clean Text
STEP 5 — Tokenize
STEP 6 — Handle Stopwords
STEP 7 — Normalize Text
STEP 8 — Validate Processed Text
STEP 9 — Interpret NLP Results
STEP 10 — Save Processed Text

---

# STEP 1 — Define NLP Objective

## Purpose
Establish the NLP preprocessing goals.

## Goals
1. Clean and normalize job description text
2. Tokenize text for analysis
3. Remove stopwords while preserving technical terms
4. Prepare text for TF-IDF and similarity analysis

## Method
Apply standard NLP preprocessing pipeline.

---

# STEP 2 — Load Job Text

## Purpose
Load the cleaned dataset for NLP processing.

## Method
Load the cleaned CSV file.

In [ ]:
# Load clean dataset
df = pd.read_csv('data/interim/sa_it_jobs_clean.csv')

print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nText columns: Job_Description, Skills")

---

# STEP 3 — Inspect Raw Text

## Purpose
Understand the raw text content before preprocessing.

## Method
Display sample text and calculate basic statistics.

In [ ]:
# Inspect raw text
print("Raw Text Inspection:")
print("="*60)

print("\nSample Job Descriptions:")
for i, desc in enumerate(df['Job_Description'].head(3), 1):
    print(f"  {i}. {desc[:150]}...")

print("\nSample Skills:")
for i, skills in enumerate(df['Skills'].head(3), 1):
    print(f"  {i}. {skills[:100]}...")

# Text length statistics
df['desc_length'] = df['Job_Description'].str.len()
print(f"\nDescription Length Statistics:")
print(df['desc_length'].describe())

---

# STEP 4 — Clean Text

## Purpose
Remove unwanted characters, HTML tags, and normalize text.

## Method
Apply text cleaning function.

In [ ]:
# Text cleaning function
def clean_text(text):
    if pd.isna(text):
        return ''
    # Convert to lowercase
    text = str(text).lower()
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove URLs
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    # Remove special characters but keep alphanumeric and spaces
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
df['Job_Description_Clean'] = df['Job_Description'].apply(clean_text)
df['Skills_Clean'] = df['Skills'].apply(clean_text)

print("Text Cleaning:")
print("="*60)
print("\nSample cleaned description:")
print(f"  {df['Job_Description_Clean'].iloc[0][:150]}...")

---

# STEP 5 — Tokenize

## Purpose
Split text into individual tokens (words).

## Method
Use simple whitespace tokenization.

In [ ]:
# Tokenization
df['desc_tokens'] = df['Job_Description_Clean'].apply(lambda x: x.split())
df['skills_tokens'] = df['Skills_Clean'].apply(lambda x: x.split())

print("Tokenization:")
print("="*60)
print(f"\nSample tokens from first job description:")
print(f"  {df['desc_tokens'].iloc[0][:15]}")
print(f"\nAverage tokens per description: {df['desc_tokens'].apply(len).mean():.1f}")

---

# STEP 6 — Handle Stopwords

## Purpose
Remove common stopwords while preserving technical terms.

## Method
Use NLTK stopwords with custom additions for job descriptions.

In [ ]:
# Download NLTK stopwords
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# Get English stopwords
stop_words = set(stopwords.words('english'))

# Add custom stopwords for job descriptions
custom_stopwords = {'job', 'role', 'position', 'candidate', 'ideal', 'working', 'work',
                    'ability', 'including', 'include', 'within', 'across', 'related',
                    'etc', 'per', 'based', 'using', 'use', 'used', 'well', 'also'}
stop_words.update(custom_stopwords)

# Remove stopwords
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words and len(token) > 2]

df['desc_tokens_clean'] = df['desc_tokens'].apply(remove_stopwords)
df['skills_tokens_clean'] = df['skills_tokens'].apply(remove_stopwords)

print("Stopword Removal:")
print("="*60)
print(f"\nSample tokens after stopword removal:")
print(f"  {df['desc_tokens_clean'].iloc[0][:15]}")
print(f"\nAverage tokens after stopword removal: {df['desc_tokens_clean'].apply(len).mean():.1f}")

---

# STEP 7 — Normalize Text

## Purpose
Apply lemmatization to reduce words to their base form.

## Method
Use NLTK WordNet lemmatizer.

In [ ]:
# Lemmatization
nltk.download('wordnet', quiet=True)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]

df['desc_tokens_lemmatized'] = df['desc_tokens_clean'].apply(lemmatize_tokens)
df['skills_tokens_lemmatized'] = df['skills_tokens_clean'].apply(lemmatize_tokens)

# Rejoin tokens into processed text
df['Job_Description_Processed'] = df['desc_tokens_lemmatized'].apply(lambda x: ' '.join(x))
df['Skills_Processed'] = df['skills_tokens_lemmatized'].apply(lambda x: ' '.join(x))

print("Lemmatization:")
print("="*60)
print(f"\nSample processed description:")
print(f"  {df['Job_Description_Processed'].iloc[0][:150]}...")

---

# STEP 8 — Validate Processed Text

## Purpose
Verify the processed text is suitable for analysis.

## Method
Check token counts and display samples.

In [ ]:
# Validation
print("Processed Text Validation:")
print("="*60)

print(f"\nAverage tokens per description (after processing): {df['desc_tokens_lemmatized'].apply(len).mean():.1f}")
print(f"Average tokens per skills (after processing): {df['skills_tokens_lemmatized'].apply(len).mean():.1f}")

print(f"\nSample processed descriptions:")
for i, desc in enumerate(df['Job_Description_Processed'].head(3), 1):
    print(f"  {i}. {desc[:100]}...")

---

# STEP 9 — Interpret NLP Results

## Result Interpretation

### What happened?
Job descriptions and skills were cleaned, tokenized, stopwords removed, and lemmatized.

### What does it mean?
The text is now in a normalized form suitable for TF-IDF vectorization and similarity analysis.

### Why does it matter?
Proper NLP preprocessing improves the quality of text-based matching and similarity calculations.

### What decision does it support?
The processed text will be used for TF-IDF and cosine similarity in Notebook 08.

### What limitation should be considered?
Some technical terms may be affected by lemmatization. Custom stopwords may remove relevant terms.

---

# STEP 10 — Save Processed Text

## Purpose
Save the processed text for use in subsequent notebooks.

## Method
Save to the interim data folder.

In [ ]:
# Save processed text
df.to_csv('data/interim/sa_it_jobs_processed.csv', index=False)
print(f"Processed dataset saved to: data/interim/sa_it_jobs_processed.csv")
print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} columns")

# TF-IDF Top Terms Visualization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Job_Description_Processed'])
feature_names = tfidf.get_feature_names_out()

# Calculate mean TF-IDF score for each term
mean_tfidf = tfidf_matrix.mean(axis=0).A1
tfidf_scores = pd.Series(mean_tfidf, index=feature_names).sort_values(ascending=False).head(20)

plt.figure(figsize=(12, 8))
sns.barplot(x=tfidf_scores.values, y=tfidf_scores.index, palette='viridis')
plt.title('Top 20 Terms by TF-IDF Score', fontsize=14, fontweight='bold')
plt.xlabel('Mean TF-IDF Score')
plt.ylabel('Term')
plt.tight_layout()
plt.savefig('outputs/nlp_tfidf_top_terms.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 20 TF-IDF Terms:")
print(tfidf_scores.to_string())

---

# FINAL NOTEBOOK RESULTS

## What We Did
Preprocessed job description text and skills through cleaning, tokenization, stopword removal, and lemmatization.

## Key Results
1. Cleaned and normalized all text fields
2. Removed stopwords while preserving technical terms
3. Applied lemmatization
4. Generated TF-IDF top terms visualization

## Result Interpretation
The text is now ready for TF-IDF vectorization and similarity analysis.

## Data Science Decisions
- Used NLTK stopwords with custom additions
- Applied WordNet lemmatization
- Preserved technical terms

## Outputs Produced
- Processed dataset: data/interim/sa_it_jobs_processed.csv
- TF-IDF visualization: outputs/nlp_tfidf_top_terms.png

## Limitations
- Lemmatization may affect some technical terms
- Custom stopwords may remove relevant terms

## Connection to the Next Notebook
Notebook 06 will use the processed text for feature engineering and statistical analysis.